H2.a

Fourrey pipeline

In [ ]:
def get_observed_categories(con, subset_filter, cat_col):
    result = con.execute(f"""
        SELECT DISTINCT CAST({cat_col} AS VARCHAR) AS cat
        FROM data
        WHERE {subset_filter} AND {cat_col} IS NOT NULL
    """).fetchdf()["cat"].tolist()
    return sorted(str(x) for x in result)

def build_full_moments_sql(con, subset_filter, all_terms, y_col):
    """
    all_terms: dict {name: sql_expression}, e.g. {"const": "1.0", "gender_M": "...", 
               "topic_bucket__X": "...", "career_age": "career_age"}
    Returns: XtX (k,k), Xty (k,), yty (float), n (int), names (list)
    """
    names = list(all_terms.keys())
    xtx_selects = [f"SUM(({all_terms[ni]}) * ({all_terms[nj]})) AS xtx_{i}_{j}"
                   for i, ni in enumerate(names) for j, nj in enumerate(names) if j >= i]
    xty_selects = [f"SUM(({all_terms[n]}) * {y_col}) AS xty_{i}" for i, n in enumerate(names)]
    yty_select = f"SUM({y_col} * {y_col}) AS yty"

    row = con.execute(f"""
        SELECT {', '.join(xtx_selects + xty_selects)}, {yty_select}, COUNT(*) AS n
        FROM data WHERE {subset_filter}
    """).fetchdf().iloc[0]

    k = len(names)
    XtX = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            if j >= i:
                XtX[i, j] = XtX[j, i] = row[f"xtx_{i}_{j}"]
    Xty = np.array([row[f"xty_{i}"] for i in range(k)])
    yty = row["yty"]
    n = int(row["n"])
    return XtX, Xty, yty, n, names

from scipy import stats

def fit_beta_means_and_pvals(con, subset_filter, all_terms, y_col):
    names = list(all_terms.keys())
    XtX, Xty, yty, n, names = build_full_moments_sql(con, subset_filter, all_terms, y_col)

    XtX_inv = np.linalg.inv(XtX)   # need the actual inverse now, not just solve()
    try:
        beta = XtX_inv @ Xty
    except np.linalg.LinAlgError:
        beta, *_ = np.linalg.lstsq(XtX, Xty, rcond=None)
        XtX_inv = np.linalg.pinv(XtX)  # fallback pseudo-inverse if singular

    beta = pd.Series(beta, index=names)
    k = len(names)

    sse = yty - beta.values @ Xty
    sigma2 = sse / (n - k)

    var_beta = sigma2 * np.diag(XtX_inv)
    se_beta = np.sqrt(var_beta)
    t_stats = beta.values / se_beta
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - k))

    se_beta = pd.Series(se_beta, index=names)
    t_stats = pd.Series(t_stats, index=names)
    p_values = pd.Series(p_values, index=names)

    const_idx = names.index("const")
    means = pd.Series(XtX[const_idx, :] / n, index=names)

    return beta, means, names, se_beta, t_stats, p_values

def fmt(x):
    return f"{x:.15e}"  # forces scientific-notation float literal -> DuckDB parses as DOUBLE

def build_fitted_view(con, subset_filter, all_terms, y_col, beta, view_name="data_fitted"):
    fitted_expr = " + ".join(
        f"({fmt(beta[name])}) * ({all_terms[name]})" for name in all_terms
    )
    con.execute(f"""
        CREATE OR REPLACE VIEW {view_name} AS
        SELECT *,
               ({fitted_expr}) AS fitted_full,
               {y_col} - ({fitted_expr}) AS resid
        FROM data
        WHERE {subset_filter}
    """)
    return view_name

def gini_sql(con, view_name, expr):
    query = f"""
        WITH ranked AS (
            SELECT (CAST(({expr}) AS DOUBLE)) AS y_s
            FROM {view_name}
        ),
        sorted AS (
            SELECT y_s, ROW_NUMBER() OVER (ORDER BY y_s) AS rnk
            FROM ranked
        )
        SELECT
            (2.0 * SUM(CAST(rnk AS DOUBLE) * y_s) - (CAST(COUNT(*) AS DOUBLE) + 1) * SUM(y_s))
            / (CAST(COUNT(*) AS DOUBLE) * SUM(y_s)) AS gini
        FROM sorted
    """
    return con.execute(query).fetchdf().iloc[0]["gini"]

def coalition_expr(all_terms, beta, means, player_groups, S):
    included_cols = set()
    for p in S:
        if p != "residual":
            included_cols.update(player_groups[p])

    excluded_terms = [name for name in all_terms if name not in included_cols and name != "const"]
# neutralize excluded terms
    correction = " - ".join(
        f"(({fmt(beta[t])}) * (({all_terms[t]}) - ({fmt(means[t])})))"
        for t in excluded_terms
    )
    expr = "fitted_full"
    if correction:
        expr = f"fitted_full - ({correction})"
    if "residual" in S:
        expr = f"({expr}) + resid"
    return expr

from itertools import combinations
from math import factorial

def fourrey_gini_shapley_sql(con, subset_filter, all_terms, y_col, player_groups, view_name="data_fitted"):
    beta, means, names, se_beta, t_stats, p_values = fit_beta_means_and_pvals(con, subset_filter, all_terms, y_col)
    build_fitted_view(con, subset_filter, all_terms, y_col, beta, view_name) # pour résoudre bug sql

    players = list(player_groups.keys()) + ["residual"]
    k = len(players)

    gini_cache = {}
    for r in range(k + 1):
        for subset in combinations(players, r):
            S = frozenset(subset)
            if len(S) == 0:
                gini_cache[S] = 0.0  # constant outcome -> zero inequality
                continue
            expr = coalition_expr(all_terms, beta, means, player_groups, S)
            gini_cache[S] = gini_sql(con, view_name, expr)
            print(f"coalition {sorted(S)}: gini = {gini_cache[S]:.5f}")  # progress, since this is slow

    shapley_values = {}
    for j in players:
        others = [p for p in players if p != j]
        phi_j = 0.0
        for r in range(len(others) + 1):
            for S in combinations(others, r):
                S = frozenset(S)
                weight = factorial(len(S)) * factorial(k - len(S) - 1) / factorial(k)
                phi_j += weight * (gini_cache[S | {j}] - gini_cache[S]) #adds contribution to every combi
        shapley_values[j] = phi_j

    total_gini = gini_cache[frozenset(players)]
    return pd.Series(shapley_values), total_gini, gini_cache

def fourrey_variance_shapley(con, subset_filter, all_terms, y_col, player_groups):
    """
    Fourrey (2023)-style Shapley decomposition of inequality, using variance 
    (= generalized entropy, theta=2) as the inequality measure and the 
    equalized game (lambda = mean). ONE model estimation; residual is an 
    explicit player. Fully computable from cached moments.
    Now reuses fit_beta_means_and_pvals for beta/means AND inference stats.
    """
    beta_full, means, names, se_beta, t_stats, p_values = fit_beta_means_and_pvals(
        con, subset_filter, all_terms, y_col
    )

    # recompute what's needed for the variance-decomposition machinery below
    XtX, Xty, yty, n, names_check = build_full_moments_sql(con, subset_filter, all_terms, y_col)
    sse = yty - beta_full.values @ Xty
    var_e = sse / n

    non_const = [nm for nm in names if nm != "const"]
    idx = [names.index(nm) for nm in non_const]
    XtX_nc = XtX[np.ix_(idx, idx)]
    mean_nc = means[non_const].values
    Cov = XtX_nc / n - np.outer(mean_nc, mean_nc)
    Cov = pd.DataFrame(Cov, index=non_const, columns=non_const)

    total_var = beta_full[non_const].values @ Cov.values @ beta_full[non_const].values + var_e

    players = list(player_groups.keys()) + ["residual"]
    k = len(players)

    def VI(S):
        v = 0.0
        cols_in_S = [c for p in S for c in player_groups.get(p, []) if p != "residual"]
        if cols_in_S:
            b = beta_full[cols_in_S].values
            C = Cov.loc[cols_in_S, cols_in_S].values
            v += b @ C @ b
        if "residual" in S:
            v += var_e
        return v

    from itertools import combinations
    from math import factorial

    var_cache = {}
    for r in range(k + 1):
        for subset in combinations(players, r):
            var_cache[frozenset(subset)] = VI(subset)

    shapley_values = {}
    for j in players:
        others = [p for p in players if p != j]
        phi_j = 0.0
        for r in range(len(others) + 1):
            for S in combinations(others, r):
                S = frozenset(S)
                weight = factorial(len(S)) * factorial(k - len(S) - 1) / factorial(k)
                phi_j += weight * (var_cache[S | {j}] - var_cache[S])
        shapley_values[j] = phi_j

    return pd.Series(shapley_values), total_var, var_cache, beta_full, se_beta, p_values

from itertools import combinations
from math import factorial

def fourrey_pmc_and_interactions(gini_cache, players):
    """
    gini_cache: dict {frozenset(subset): gini_value}, already computed
    players: list of all player names (including 'residual')
    Returns: pmc (pd.Series), interactions (pd.DataFrame, symmetric, diagonal = PMC)
    """
    m = len(players)
    full_set = frozenset(players)

    # ---- Pure Marginal Contribution: VI(M) - VI(M \ {i}) ----
    pmc = {}
    for i in players:
        M_minus_i = full_set - {i}
        pmc[i] = gini_cache[full_set] - gini_cache[M_minus_i]
    pmc = pd.Series(pmc)

    # ---- Pairwise interaction Int(i,j,T) for every T, then weighted average INT(i,j) ----
    interactions = pd.DataFrame(0.0, index=players, columns=players)

    for i, j in combinations(players, 2):
        others = [p for p in players if p not in (i, j)]
        INT_ij = 0.0
        for t in range(len(others) + 1):
            for T in combinations(others, t):
                T = frozenset(T)
                weight = factorial(m - t - 2) * factorial(t + 1) / factorial(m)
                Int_T = (gini_cache[T | {i, j}] - gini_cache[T | {i}]
                         - gini_cache[T | {j}] + gini_cache[T])
                INT_ij += weight * Int_T
        interactions.loc[i, j] = INT_ij
        interactions.loc[j, i] = INT_ij

    for i in players:
        interactions.loc[i, i] = pmc[i]

    # sanity check: SHAP_i = PMC_i - sum_j INT(i,j)  should equal shapley_vals[i]
    shap_reconstructed = pmc - interactions.drop(columns=[]).apply(
        lambda row: sum(interactions.loc[row.name, j] for j in players if j != row.name), axis=1
    )

    return pmc, interactions, shap_reconstructed

In [ ]:
import duckdb

# note: this only works if the parquet has been read into a table/view first,
# since information_schema/duckdb_columns() need a registered relation, not a raw file path
duckdb.sql("CREATE OR REPLACE VIEW a2t AS SELECT * EXCLUDE(field_name,field_pop,field_computer_science) FROM 'a2t_dec_final.parquet'")

field_cols = duckdb.sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'a2t'
      AND column_name LIKE 'field_%'
""").df()["column_name"].tolist()

In [ ]:
# prod, gini
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(prod_3 + 1) AS log_prod,
    FROM read_parquet('a2t_dec_final.parquet')
""")


all_terms = {"const": "1.0", "gender": "CASE WHEN gender = 'M' THEN 1.0 ELSE 0.0 END"}

# dummies
for cat in get_observed_categories(con, "1=1", "cohort")[1:]:
    all_terms[f"cohort__{cat}"] = f"CASE WHEN CAST(cohort AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "first_prod")[1:]:
    all_terms[f"first__{cat}"] = f"CASE WHEN CAST(first_prod AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "insti_rank")[1:]:
    all_terms[f"insti__{cat}"] = f"CASE WHEN CAST(insti_rank AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"

#cont var
all_terms["career_age"] = "career_age"
#all_terms["n_unique_coauthors_7y"] = "n_unique_coauthors_7y"
#all_terms["field_pop"] = "field_pop"
all_terms["gini_field"] = "gini_field"

for field in field_cols : 
    all_terms[field] = field

# player=variables
player_groups = {
    "gender": ["gender"],
    #"topic": [nm for nm in all_terms if nm.startswith("topic__")],
    "cohort": [nm for nm in all_terms if nm.startswith("cohort__")],
    "first": [nm for nm in all_terms if nm.startswith("first__")],
    "career_age" : ["career_age"],
    "gini_field" : ["gini_field"],
    #"field_pop" : ["field_pop"],
    "topic" : field_cols,
    #"n_unique_coauthors_7y" : ["n_unique_coauthors_7y"],
    "insti_rank" : [nm for nm in all_terms if nm.startswith("insti__")]
}

shapley_vals, total_gini, gini_cache = fourrey_gini_shapley_sql(con, "1=1", all_terms, "log_prod", player_groups)


print(shapley_vals / total_gini * 100)
print("sum:", shapley_vals.sum(), "total gini:", total_gini)


players = list(player_groups.keys()) + ["residual"]
pmc, interactions, shap_check = fourrey_pmc_and_interactions(gini_cache, players)

print("PMC (as % of total gini):")
print(pmc / total_gini * 100)

print("\nPairwise interactions (as % of total gini), off-diagonal, diagonal = PMC:")
print((interactions / total_gini * 100).round(2))

print("\nSanity check — should match your earlier shapley_vals:")
print(shap_check)

In [ ]:
# prod, var
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(cite_3 + 1) AS log_cite, LN(prod_3 + 1) AS log_prod,
    FROM read_parquet('a2t_dec_final.parquet')
""")

all_terms = {"const": "1.0", "gender": "CASE WHEN gender = 'M' THEN 1.0 ELSE 0.0 END"}

# dummies
for cat in get_observed_categories(con, "1=1", "cohort")[1:]:
    all_terms[f"cohort__{cat}"] = f"CASE WHEN CAST(cohort AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "first_prod")[1:]:
    all_terms[f"first__{cat}"] = f"CASE WHEN CAST(first_prod AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "insti_rank")[1:]:
    all_terms[f"insti__{cat}"] = f"CASE WHEN CAST(insti_rank AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"

#cont var
all_terms["career_age"] = "career_age"
#all_terms["n_unique_coauthors_7y"] = "n_unique_coauthors_7y"
#all_terms["field_pop"] = "field_pop"
all_terms["gini_field"] = "gini_field"


for field in field_cols : 
    all_terms[field] = field

# player=variables
player_groups = {
    "gender": ["gender"],
    #"topic": [nm for nm in all_terms if nm.startswith("topic__")],
    "cohort": [nm for nm in all_terms if nm.startswith("cohort__")],
    "first": [nm for nm in all_terms if nm.startswith("first__")],
    "career_age" : ["career_age"],
    "gini_field" : ["gini_field"],
    #"field_pop" : ["field_pop"],
    "topic" : field_cols,
    #"n_unique_coauthors_7y" : ["n_unique_coauthors_7y"],
    "insti_rank" : [nm for nm in all_terms if nm.startswith("insti__")]
}

shapley_vals, total_var, var_cache, betas, se, pval = fourrey_variance_shapley(con, "1=1", all_terms, "log_prod", player_groups)

print("Original model: ")
print(pd.DataFrame({"coef" : betas, "se" : se, "p-val" : pval}))

print(shapley_vals / total_var * 100)
print("sum:", shapley_vals.sum(), "total gini:", total_var)


players = list(player_groups.keys()) + ["residual"]
pmc, interactions, shap_check = fourrey_pmc_and_interactions(var_cache, players)

print("PMC (as % of total var):")
print(pmc / total_var * 100)

print("\nPairwise interactions (as % of total var), off-diagonal, diagonal = PMC:")
print((interactions / total_var * 100).round(2))

print("\nSanity check — should match your earlier shapley_vals:")
print(shap_check)

In [ ]:
# cite, gini
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(cite_3 + 1) AS log_cite, LN(prod_3 + 1) AS log_prod,
    FROM read_parquet('a2t_dec_final.parquet')
""")


all_terms = {"const": "1.0", "gender": "CASE WHEN gender = 'M' THEN 1.0 ELSE 0.0 END"}

# dummies
for cat in get_observed_categories(con, "1=1", "cohort")[1:]:
    all_terms[f"cohort__{cat}"] = f"CASE WHEN CAST(cohort AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "first_cite")[1:]:
    all_terms[f"first__{cat}"] = f"CASE WHEN CAST(first_prod AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "insti_rank")[1:]:
    all_terms[f"insti__{cat}"] = f"CASE WHEN CAST(insti_rank AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"

#cont var
all_terms["career_age"] = "career_age"
#all_terms["n_unique_coauthors_7y"] = "n_unique_coauthors_7y"
#all_terms["field_pop"] = "field_pop"
all_terms["gini_field"] = "gini_field"
all_terms["log_prod"] = "log_prod"

for field in field_cols : 
    all_terms[field] = field

# player=variables
player_groups = {
    "gender": ["gender"],
    #"topic": [nm for nm in all_terms if nm.startswith("topic__")],
    "cohort": [nm for nm in all_terms if nm.startswith("cohort__")],
    "first": [nm for nm in all_terms if nm.startswith("first__")],
    "career_age" : ["career_age"],
    "gini_field" : ["gini_field"],
    #"field_pop" : ["field_pop"],
    "topic" : field_cols,
    #"n_unique_coauthors_7y" : ["n_unique_coauthors_7y"],
    "insti_rank" : [nm for nm in all_terms if nm.startswith("insti__")],
    "log_prod" : ["log_prod"]
}

shapley_vals, total_gini, gini_cache = fourrey_gini_shapley_sql(con, "1=1", all_terms, "log_cite", player_groups)


print(shapley_vals / total_gini * 100)
print("sum:", shapley_vals.sum(), "total gini:", total_gini)


players = list(player_groups.keys()) + ["residual"]
pmc, interactions, shap_check = fourrey_pmc_and_interactions(gini_cache, players)

print("PMC (as % of total gini):")
print(pmc / total_gini * 100)

print("\nPairwise interactions (as % of total gini), off-diagonal, diagonal = PMC:")
print((interactions / total_gini * 100).round(2))

print("\nSanity check — should match your earlier shapley_vals:")
print(shap_check)

In [ ]:
# cite, var
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(cite_3 + 1) AS log_cite, LN(prod_3 + 1) AS log_prod,
    FROM read_parquet('a2t_dec_final.parquet')
""")

all_terms = {"const": "1.0", "gender": "CASE WHEN gender = 'M' THEN 1.0 ELSE 0.0 END"}

# dummies
for cat in get_observed_categories(con, "1=1", "cohort")[1:]:
    all_terms[f"cohort__{cat}"] = f"CASE WHEN CAST(cohort AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "first_cite")[1:]:
    all_terms[f"first__{cat}"] = f"CASE WHEN CAST(first_cite AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"
for cat in get_observed_categories(con, "1=1", "insti_rank")[1:]:
    all_terms[f"insti__{cat}"] = f"CASE WHEN CAST(insti_rank AS VARCHAR) = '{cat}' THEN 1.0 ELSE 0.0 END"

#cont var
all_terms["career_age"] = "career_age"
#all_terms["n_unique_coauthors_7y"] = "n_unique_coauthors_7y"
#all_terms["field_pop"] = "field_pop"
all_terms["gini_field"] = "gini_field"
all_terms["log_prod"] = "log_prod"


for field in field_cols : 
    all_terms[field] = field

# player=variables
player_groups = {
    "gender": ["gender"],
    #"topic": [nm for nm in all_terms if nm.startswith("topic__")],
    "cohort": [nm for nm in all_terms if nm.startswith("cohort__")],
    "first": [nm for nm in all_terms if nm.startswith("first__")],
    "career_age" : ["career_age"],
    "gini_field" : ["gini_field"],
    #"field_pop" : ["field_pop"],
    "topic" : field_cols,
    #"n_unique_coauthors_7y" : ["n_unique_coauthors_7y"],
    "insti_rank" : [nm for nm in all_terms if nm.startswith("insti__")],
    "log_prod" : ["log_prod"]
}

shapley_vals, total_var, var_cache, betas, se, pval = fourrey_variance_shapley(con, "1=1", all_terms, "log_cite", player_groups)

print("Original model: ")
print(pd.DataFrame({"coef" : betas, "se" : se, "p-val" : pval}))

print(shapley_vals / total_var * 100)
print("sum:", shapley_vals.sum(), "total gini:", total_var)


players = list(player_groups.keys()) + ["residual"]
pmc, interactions, shap_check = fourrey_pmc_and_interactions(var_cache, players)

print("PMC (as % of total var):")
print(pmc / total_var * 100)

print("\nPairwise interactions (as % of total var), off-diagonal, diagonal = PMC:")
print((interactions / total_var * 100).round(2))

print("\nSanity check — should match your earlier shapley_vals:")
print(shap_check)

H2.b

Kim pipeline

In [ ]:
def yun_normalize(beta_full, shares_full):
    """
    beta_full: dict {category_name: raw_coef}, including the reference 
               category with coef = 0 (since it was dropped in OLS)
    shares_full: dict {category_name: sample_share}, same keys, sums to 1
    Returns: (normalized_betas dict, intercept_adjustment)
    """
    weighted_mean = sum(shares_full[k] * beta_full[k] for k in beta_full)
    normalized = {k: beta_full[k] - weighted_mean for k in beta_full}
    intercept_adjustment = weighted_mean
    return normalized, intercept_adjustment

def get_observed_categories(con, subset_filter, cat_col):
    result = con.execute(f"""
        SELECT DISTINCT CAST({cat_col} AS VARCHAR) AS cat
        FROM data
        WHERE {subset_filter} AND {cat_col} IS NOT NULL
    """).fetchdf()["cat"].tolist()
    return sorted(str(x) for x in result)

def fit_and_normalize(con, subset_filter, cat_cols, cont_cols, y_col):
    """
    Generalized version: handles multiple categorical variables (e.g. topic, cohort),
    each Yun-normalized independently, plus continuous controls.

    cat_cols: list of categorical column names, e.g. ["topic_bucket", "cohort"]
    all_categories_dict: dict {cat_col: list_of_all_categories}, e.g.
                          {"topic_bucket": all_topics, "cohort": all_cohorts}
    Returns: beta_0_adj (float), beta_star (pd.Series, index = "topic_bucket_X", 
             "cohort_Y", cont_col names), X_star (pd.Series, same index)
    """
    # getting categories of categorical vars
    all_categories_dict = {}
    for cat_col in cat_cols : 
        all_categories_dict[cat_col] = get_observed_categories(con,subset_filter,cat_col)

    # ---- 1. Build dummy expressions for EVERY categorical var, K_c - 1 dummies each ----
    all_terms = {"const": "1.0"} # intercept
    ref_categories = {}       # {cat_col: reference category}
    dummy_var_categories = {} # {cat_col: [dummy categories, i.e. all minus reference]}

    for cat_col in cat_cols:
        cats = all_categories_dict[cat_col]
        ref_categories[cat_col] = cats[0]
        dummy_var_categories[cat_col] = cats[1:]
        for cat in cats[1:]:
            key = f"{cat_col}__{cat}"   # unique name across all cat vars
            all_terms[key] = f"CASE WHEN {cat_col} = '{cat}' THEN 1.0 ELSE 0.0 END" # dummies creation

    for c in cont_cols:
        all_terms[c] = c

    names = list(all_terms.keys())

    # ---- 2. X'X, X'y via SQL (unchanged logic, just more terms now) ----
    xtx_selects = []
    for i, ni in enumerate(names):
        for j, nj in enumerate(names):
            if j >= i:
                xtx_selects.append(f"SUM(({all_terms[ni]}) * ({all_terms[nj]})) AS xtx_{i}_{j}")
    xty_selects = [f"SUM(({all_terms[n]}) * {y_col}) AS xty_{i}" for i, n in enumerate(names)]

    query = f"""
        SELECT {', '.join(xtx_selects + xty_selects)}, COUNT(*) AS n
        FROM data
        WHERE {subset_filter}
    """
    row = con.execute(query).fetchdf().iloc[0]

    k = len(names)
    XtX = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            if j >= i:
                XtX[i, j] = XtX[j, i] = row[f"xtx_{i}_{j}"]
    Xty = np.array([row[f"xty_{i}"] for i in range(k)])
    n = int(row["n"])

    # solves, no need for inverse
    beta = np.linalg.solve(XtX, Xty)
    beta_series = pd.Series(beta, index=names)
    beta_0 = beta_series["const"]

    # ---- 3. Per-variable category shares (separate query per cat_col) ----
    shares_by_var = {}  # {cat_col: {category: share}}
    for cat_col in cat_cols:
        shares_query = f"""
            SELECT {cat_col} AS cat, COUNT(*) * 1.0 / {n} AS share
            FROM data
            WHERE {subset_filter}
            GROUP BY {cat_col}
        """
        shares_df = con.execute(shares_query).fetchdf()
        shares_full = dict(zip(shares_df["cat"].astype(str), shares_df["share"]))
        for cat in all_categories_dict[cat_col]:
            shares_full.setdefault(str(cat), 0.0)
        shares_by_var[cat_col] = shares_full

    # ---- 4. Yun-normalize EACH categorical block independently ----
    beta_0_adj = beta_0
    beta_star_parts = []
    xstar_parts = []

    for cat_col in cat_cols:
        ref_cat = ref_categories[cat_col]
        beta_full = {str(ref_cat): 0.0}
        for cat in dummy_var_categories[cat_col]:
            beta_full[str(cat)] = beta_series[f"{cat_col}__{cat}"]

        shares_full = shares_by_var[cat_col]
        beta_norm_cat, intercept_adj = yun_normalize(beta_full, shares_full)
        beta_0_adj += intercept_adj   # each categorical block contributes its own mean to intercept

        # prefix keys with cat_col so they don't collide across variables in beta_star's index
        beta_star_parts.append(pd.Series({f"{cat_col}__{k}": v for k, v in beta_norm_cat.items()}))
        xstar_parts.append(pd.Series({f"{cat_col}__{k}": v for k, v in shares_full.items()}))

    # ---- 5. Continuous means via SQL ----
    if cont_cols:
        cont_means_query = f"""
            SELECT {', '.join(f'AVG({c}) AS {c}' for c in cont_cols)}
            FROM data
            WHERE {subset_filter}
        """
        cont_means_row = con.execute(cont_means_query).fetchdf().iloc[0]
        xstar_cont = pd.Series({c: cont_means_row[c] for c in cont_cols})
        beta_star_cont = beta_series[cont_cols]
    else:
        xstar_cont = pd.Series(dtype=float)
        beta_star_cont = pd.Series(dtype=float)
    # computes beta* (without intercept and normalized) and X* (mean without intercept)
    beta_star = pd.concat(beta_star_parts + [beta_star_cont])
    X_star = pd.concat(xstar_parts + [xstar_cont])

    return beta_0_adj, beta_star, X_star

def kim_decomposition(beta0_At, betastar_At, xstar_At,
                       beta0_As, betastar_As, xstar_As,
                       beta0_Bt, betastar_Bt, xstar_Bt,
                       beta0_Bs, betastar_Bs, xstar_Bs):

    # D1: intercept difference-in-differences
    D1 = (beta0_At - beta0_As) - (beta0_Bt - beta0_Bs)

    # D2: pure coefficient effect (grand mean X, double-diff in coefficients)
    X_grand_mean = (xstar_At + xstar_As + xstar_Bt + xstar_Bs) / 4
    coef_diff_diff = (betastar_At - betastar_As) - (betastar_Bt - betastar_Bs)
    D2 = X_grand_mean @ coef_diff_diff

    # D3: coefficient interaction effect (avg endowment change x level diff in avg coefs)
    avg_endow_change = ((xstar_At - xstar_As) + (xstar_Bt - xstar_Bs)) / 2
    avg_coef_level_diff = (betastar_At + betastar_As) / 2 - (betastar_Bt + betastar_Bs) / 2
    D3 = avg_endow_change @ avg_coef_level_diff

    # D4: pure endowment effect (double-diff in endowments x grand mean coefs)
    beta_grand_mean = (betastar_At + betastar_As + betastar_Bt + betastar_Bs) / 4
    endow_diff_diff = (xstar_At - xstar_As) - (xstar_Bt - xstar_Bs)
    D4 = endow_diff_diff @ beta_grand_mean

    # D5: endowment interaction effect (level diff in avg endowments x avg coef change)
    avg_endow_level_diff = (xstar_At + xstar_As) / 2 - (xstar_Bt + xstar_Bs) / 2
    avg_coef_change = ((betastar_At - betastar_As) + (betastar_Bt - betastar_Bs)) / 2
    D5 = avg_endow_level_diff @ avg_coef_change

    delta_gap = D1 + D2 + D3 + D4 + D5

    return {"D1_intercept": D1, "D2_pure_coefficient": D2,
            "D3_coefficient_interaction": D3, "D4_pure_endowment": D4,
            "D5_endowment_interaction": D5, "delta_gap_total": delta_gap}

def get_kim_factors(cache, group_a, group_b, time_s, time_t):
    beta0_At, betastar_At, xstar_At = cache[(group_a, time_t)]
    beta0_As, betastar_As, xstar_As = cache[(group_a, time_s)]
    beta0_Bt, betastar_Bt, xstar_Bt = cache[(group_b, time_t)]
    beta0_Bs, betastar_Bs, xstar_Bs = cache[(group_b, time_s)]

    full_index = betastar_At.index
    for s in [betastar_As, betastar_Bt, betastar_Bs]:
        full_index = full_index.union(s.index)
    align = lambda s: s.reindex(full_index, fill_value=0.0)
    betastar_At, betastar_As = align(betastar_At), align(betastar_As)
    betastar_Bt, betastar_Bs = align(betastar_Bt), align(betastar_Bs)
    xstar_At, xstar_As = align(xstar_At), align(xstar_As)
    xstar_Bt, xstar_Bs = align(xstar_Bt), align(xstar_Bs)

    # D3 factors: avg endowment CHANGE, avg coefficient LEVEL-DIFF
    avg_endow_change = ((xstar_At - xstar_As) + (xstar_Bt - xstar_Bs)) / 2
    avg_coef_level_diff = (betastar_At + betastar_As)/2 - (betastar_Bt + betastar_Bs)/2

    # D5 factors: avg endowment LEVEL-DIFF, avg coefficient CHANGE
    avg_endow_level_diff = (xstar_At + xstar_As)/2 - (xstar_Bt + xstar_Bs)/2
    avg_coef_change = ((betastar_At - betastar_As) + (betastar_Bt - betastar_Bs)) / 2

    return {
        "D3_endow_factor": avg_endow_change.sum(),
        "D3_coef_factor": avg_coef_level_diff.sum(),
        "D5_endow_factor": avg_endow_level_diff.sum(),
        "D5_coef_factor": avg_coef_change.sum(),
    }

def fit_all_cells(con, y_col, cat_cols, cont_cols, group_col, group_a, group_b, time_col, all_years):
    # all (group x year) pairs
    cache = {}
    for g in [group_a, group_b]:
        for t in all_years:
            filt = f"{group_col} = '{g}' AND {time_col} = {t}"
            cache[(g, t)] = fit_and_normalize(con, filt, cat_cols, cont_cols, y_col)
    return cache

def align_series(s, full_index):
    return s.reindex(full_index, fill_value=0.0)

def kim_from_cache(cache, group_a, group_b, time_s, time_t):
    beta0_At, betastar_At, xstar_At = cache[(group_a, time_t)]
    beta0_As, betastar_As, xstar_As = cache[(group_a, time_s)]
    beta0_Bt, betastar_Bt, xstar_Bt = cache[(group_b, time_t)]
    beta0_Bs, betastar_Bs, xstar_Bs = cache[(group_b, time_s)]

    full_index = betastar_At.index
    for s in [betastar_As, betastar_Bt, betastar_Bs]:
        full_index = full_index.union(s.index)
        
    # empty space where rows do not exist due to group x time extraction
    betastar_At, xstar_At = align_series(betastar_At, full_index), align_series(xstar_At, full_index)
    betastar_As, xstar_As = align_series(betastar_As, full_index), align_series(xstar_As, full_index)
    betastar_Bt, xstar_Bt = align_series(betastar_Bt, full_index), align_series(xstar_Bt, full_index)
    betastar_Bs, xstar_Bs = align_series(betastar_Bs, full_index), align_series(xstar_Bs, full_index)

    return kim_decomposition(beta0_At, betastar_At, xstar_At,
                              beta0_As, betastar_As, xstar_As,
                              beta0_Bt, betastar_Bt, xstar_Bt,
                              beta0_Bs, betastar_Bs, xstar_Bs)

def sanity_check(con, y_col, group_col, group_a, group_b, time_col, time_s, time_t):
    # overall mean change between t and s
    
    sanity_query = f"""
        SELECT
            AVG(CASE WHEN {group_col} = '{group_a}' AND {time_col} = {time_t} THEN {y_col} END) AS mean_At,
            AVG(CASE WHEN {group_col} = '{group_b}' AND {time_col} = {time_t} THEN {y_col} END) AS mean_Bt,
            AVG(CASE WHEN {group_col} = '{group_a}' AND {time_col} = {time_s} THEN {y_col} END) AS mean_As,
            AVG(CASE WHEN {group_col} = '{group_b}' AND {time_col} = {time_s} THEN {y_col} END) AS mean_Bs
        FROM data
    """
    means = con.execute(sanity_query).fetchdf().iloc[0]
    gap_t = means["mean_At"] - means["mean_Bt"]
    gap_s = means["mean_As"] - means["mean_Bs"]
    return gap_t - gap_s

def run_kim_decomposition(con, y_col, cat_cols, cont_cols, group_col, group_a, group_b,
                            time_col, time_s, time_t):
    cache = fit_all_cells(con, y_col, cat_cols, cont_cols, group_col, group_a, group_b,
                           time_col, [time_s, time_t])
    result = kim_from_cache(cache, group_a, group_b, time_s, time_t)
    result["actual_delta_gap"] = sanity_check(con, y_col, group_col, group_a, group_b, time_col, time_s, time_t)
    return result

In [ ]:
import duckdb

# note: this only works if the parquet has been read into a table/view first,
# since information_schema/duckdb_columns() need a registered relation, not a raw file path
duckdb.sql("CREATE OR REPLACE VIEW a2t AS SELECT * EXCLUDE(field_name,field_pop,field_computer_science) FROM 'a2t_dec_final.parquet'")

field_cols = duckdb.sql("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'a2t'
      AND column_name LIKE 'field_%'
""").df()["column_name"].tolist()

print(field_cols)

In [ ]:
# model 1 log_prod
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(prod_3 + 1) AS log_prod,
    FROM read_parquet('a2t_dec_final.parquet')
""")

models = {
    "Model_1" : [["cohort"],[]], #demo
    "Model_2" : [["cohort"],["gini_field","share_3"]], #human capi
    "Model_3" : [["cohort","first_prod"],["gini_field","share_3"]], #human capi
    "Model_4" : [["cohort","first_prod"],["gini_field","share_3"]+field_cols], #topics
    "Model_5" : [["cohort","first_prod","insti_rank"],["gini_field","share_3"]+field_cols] #job context
}


all_years = list(range(1990, 2024))
res = pd.DataFrame()
factor_rows = []

for model, cov in models.items():
    cache = fit_all_cells(con, "log_prod", cov[0], cov[1], "gender", "M", "F", "year", all_years)

    for t in all_years[:-1]:
        dec = kim_from_cache(cache, "M", "F", t, t+1)
        dec["actual_delta_gap"] = sanity_check(con, "log_prod", "gender", "M", "F", "year", t, t+1)

        temp = pd.DataFrame([dec])
        temp["year"] = f"{t}-{t+1}"
        temp["model"] = model
        res = pd.concat([res, temp])

        factors = get_kim_factors(cache, "M", "F", t, t+1)
        factors["year"] = f"{t}-{t+1}"
        factors["model"] = model
        factor_rows.append(factors)

    print(f"{model} finished.")

res.to_csv("kim_log_prod.csv", index=False)
pd.DataFrame(factor_rows).to_csv("kim_log_prod_factors.csv", index=False)

In [ ]:
# model 1 log_cite
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(prod_3 + 1) AS log_prod, LN(cite_3 + 1) AS log_cite,
    FROM read_parquet('a2t_dec_final.parquet')
""")

models = {
    "Model_1" : [["cohort"],[]], #demo
    "Model_2" : [["cohort"],["gini_field","share_3"]], #human capi
    "Model_3" : [["cohort","first_cite"],["gini_field","share_3","log_prod"]],
    "Model_4" : [["cohort","first_cite"],["gini_field","share_3","log_prod"]+field_cols], #topics
    "Model_5" : [["cohort","first_cite","insti_rank"],["gini_field","share_3","log_prod"]+field_cols] #job context
}


all_years = list(range(1990, 2024))
res = pd.DataFrame()
factor_rows = []

for model, cov in models.items():
    cache = fit_all_cells(con, "log_cite", cov[0], cov[1], "gender", "M", "F", "year", all_years)

    for t in all_years[:-1]:
        dec = kim_from_cache(cache, "M", "F", t, t+1)
        dec["actual_delta_gap"] = sanity_check(con, "log_cite", "gender", "M", "F", "year", t, t+1)

        temp = pd.DataFrame([dec])
        temp["year"] = f"{t}-{t+1}"
        temp["model"] = model
        res = pd.concat([res, temp])

        factors = get_kim_factors(cache, "M", "F", t, t+1)
        factors["year"] = f"{t}-{t+1}"
        factors["model"] = model
        factor_rows.append(factors)

    print(f"{model} finished.")

res.to_csv("kim_log_cite.csv", index=False)
pd.DataFrame(factor_rows).to_csv("kim_log_cite_factors.csv", index=False)

Graphs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

exclude_cols = ["year", "model", "actual_delta_gap", "year_num"]
plot_cols = [c for c in res.columns if c not in exclude_cols]

res["year_num"] = res["year"].str.split("-").str[0].astype(int)

n_cols = 2
n_rows = -(-len(plot_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows), squeeze=False)
axes = axes.flatten()

models_list = res["model"].unique()
n_models = len(models_list)
jitter_span = 0.3
jitters = np.linspace(-jitter_span/2, jitter_span/2, n_models) if n_models > 1 else [0]

for i, col in enumerate(plot_cols):
    if col == "delta_gap_total" : 
        ax = axes[i]
        sub = res[res["model"] == "Model_1"].sort_values("year_num")
        ax.plot(sub["year_num"], sub[col], marker="o", label="sum of D", alpha=0.85)

        ax.axhline(0, color="black", linestyle="-", linewidth=0.8)
        ax.set_xlabel("Year")
        ax.set_ylabel(col)
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=0)
        ax.legend(title="Model", fontsize=8)
    else : 
        ax = axes[i]
        for model, jit in zip(models_list, jitters):
            sub = res[res["model"] == model].sort_values("year_num")
            ax.plot(sub["year_num"] + jit, sub[col], marker="o", label=model, alpha=0.85)

        ax.axhline(0, color="black", linestyle="-", linewidth=0.8)
        ax.set_xlabel("Year")
        ax.set_ylabel(col)
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=0)
        ax.legend(title="Model", fontsize=8)

for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

Annex

Factor decomposition

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

factors_df["year_num"] = factors_df["year"].str.split("-").str[0].astype(int)

factor_cols = ["D3_endow_factor", "D3_coef_factor", "D5_endow_factor", "D5_coef_factor"]
models_list = factors_df["model"].unique()

n_cols = 2
n_rows = -(-len(factor_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 4*n_rows), squeeze=False)
axes = axes.flatten()

n_models = len(models_list)
jitter_span = 0.3
jitters = np.linspace(-jitter_span/2, jitter_span/2, n_models) if n_models > 1 else [0]

for i, col in enumerate(factor_cols):
    ax = axes[i]
    for model, jit in zip(models_list, jitters):
        sub = factors_df[factors_df["model"] == model].sort_values("year_num")
        ax.plot(sub["year_num"] + jit, sub[col], marker="o", label=model, alpha=0.85)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Year")
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title="Model", fontsize=8)

for j in range(len(factor_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

Omission bias check

In [ ]:
def compare_coefs_across_models(con, models_to_compare, y_col, group_col, group_val, year, cont_cols_common=None):
    """
    models_to_compare: dict of {model_name: (cat_cols, cont_cols)} -- same shape as your `models` dict
    Compares beta_star for variables common across the given models, for one (group, year) cell.
    """
    filt = f"{group_col} = '{group_val}' AND year = {year}"
    results = {}
    for name, (cat_cols, cont_cols) in models_to_compare.items():
        beta_0, beta_star, x_star = fit_and_normalize(con, filt, cat_cols, cont_cols, y_col)
        results[name] = beta_star

    # align on the union of all indices, fill missing with NaN (not 0 -- these are different models,
    # a variable absent from a model just means "not estimated here", not "zero effect")
    all_idx = results[list(results.keys())[0]].index
    for s in list(results.values())[1:]:
        all_idx = all_idx.union(s.index)
    aligned = {name: s.reindex(all_idx) for name, s in results.items()}

    comparison = pd.DataFrame(aligned)
    # only keep variables present in at least 2 models being compared, for a meaningful comparison
    comparison = comparison.dropna(thresh=2)
    return comparison

In [ ]:
con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(prod_3 + 1) AS log_prod, LN(cite_3 + 1) AS log_cite,
    FROM read_parquet('a2t_dec_final.parquet')
""")


models = {
    "Model_1" : [["cohort"],[]], #demo
    "Model_2" : [["cohort"],["gini_field","share_3"]], #human capi
    "Model_3" : [["cohort","first_cite"],["gini_field","share_3","log_prod"]],
    "Model_4" : [["cohort","first_cite"],["gini_field","share_3","log_prod"]+field_cols], #topics
    "Model_5" : [["cohort","first_cite","insti_rank"],["gini_field","share_3","log_prod"]+field_cols] #job context
}

om_test = pd.DataFrame(columns=["comp","min","max","mean"])
for comp in [["Model_1","Model_2"],["Model_2","Model_3"],["Model_3","Model_4"],["Model_4","Model_5"]] : 
    all_comparisons = []
    for g in ["M", "F"]:
        for year in [1995, 2005, 2015, 2020]:
            c = compare_coefs_across_models(con,
                {comp[0]: (models[comp[0]][0], models[comp[0]][1]),
                comp[1]: (models[comp[1]][0], models[comp[1]][1])},
                y_col="log_cite", group_col="gender", group_val=g, year=year
            )
            c["gender"] = g
            c["year"] = year
            c["change"] = (c[comp[1]] - c[comp[0]])/ c[comp[0]].abs()*100
            all_comparisons.append(c)

    full_comparison = pd.concat(all_comparisons)
    om_test = pd.concat([om_test,pd.DataFrame([[f"{comp[0]}→{comp[1]}",full_comparison["change"].min(),full_comparison["change"].max(),full_comparison["change"].mean()]],columns=om_test.columns)],ignore_index=True)

In [ ]:
con = duckdb.connect()
con.execute("PRAGMA disable_progress_bar")
con.execute("""
    CREATE OR REPLACE VIEW data AS
    SELECT *, LN(prod_3 + 1) AS log_prod,
    FROM read_parquet('a2t_dec_final.parquet')
""")

models = {
    "Model_1" : [["cohort"],[]], #demo
    "Model_2" : [["cohort"],["gini_field","share_3"]], #human capi
    "Model_3" : [["cohort","first_prod"],["gini_field","share_3"]], #human capi
    "Model_4" : [["cohort","first_prod"],["gini_field","share_3"]+field_cols], #topics
    "Model_5" : [["cohort","first_prod","insti_rank"],["gini_field","share_3"]+field_cols] #job context
}

om_test = pd.DataFrame(columns=["comp","min","max","mean"])
for comp in [["Model_1","Model_2"],["Model_2","Model_3"],["Model_3","Model_4"],["Model_4","Model_5"]] : 
    all_comparisons = []
    for g in ["M", "F"]:
        for year in [1995, 2005, 2015, 2020]:
            c = compare_coefs_across_models(con,
                {comp[0]: (models[comp[0]][0], models[comp[0]][1]),
                comp[1]: (models[comp[1]][0], models[comp[1]][1])},
                y_col="log_prod", group_col="gender", group_val=g, year=year
            )
            c["gender"] = g
            c["year"] = year
            c["change"] = (c[comp[1]] - c[comp[0]])/ c[comp[0]].abs()*100
            all_comparisons.append(c)

    full_comparison = pd.concat(all_comparisons)
    om_test = pd.concat([om_test,pd.DataFrame([[f"{comp[0]}→{comp[1]}",full_comparison["change"].min(),full_comparison["change"].max(),full_comparison["change"].mean()]],columns=om_test.columns)],ignore_index=True)

In [ ]:
import pandas as pd

res = pd.read_csv("kim_log_prod.csv")
factors_df = pd.read_csv("kim_log_prod_factors.csv")

In [ ]:
merged = res.merge(factors_df, on=["year", "model"])
models_list = factors_df["model"].unique()

for model in models_list:
    sub = merged[merged["model"] == model]
    print(model)
    print("  D3 vs endow_factor:", sub["D3_coefficient_interaction"].corr(sub["D3_endow_factor"]))
    print("  D3 vs coef_factor: ", sub["D3_coefficient_interaction"].corr(sub["D3_coef_factor"]))
    print("  D5 vs endow_factor:", sub["D5_endowment_interaction"].corr(sub["D5_endow_factor"]))
    print("  D5 vs coef_factor: ", sub["D5_endowment_interaction"].corr(sub["D5_coef_factor"]))

In [ ]:
merged = res.merge(factors_df, on=["year", "model"])
models_list = factors_df["model"].unique()

for model in models_list:
    sub = merged[merged["model"] == model]
    print(model)
    print("  D3 vs endow_factor:", sub["D3_coefficient_interaction"].corr(sub["D3_endow_factor"]))
    print("  D3 vs coef_factor: ", sub["D3_coefficient_interaction"].corr(sub["D3_coef_factor"]))
    print("  D5 vs endow_factor:", sub["D5_endowment_interaction"].corr(sub["D5_endow_factor"]))
    print("  D5 vs coef_factor: ", sub["D5_endowment_interaction"].corr(sub["D5_coef_factor"]))

Contribution decomposition 

In [ ]:
def decompose_product_change(factors_df, factor_a_col, factor_b_col, group_col="model"):
    df = factors_df.sort_values("year").copy()
    df["A_lag"] = df.groupby(group_col)[factor_a_col].shift(1)
    df["B_lag"] = df.groupby(group_col)[factor_b_col].shift(1)

    df["delta_A"] = df[factor_a_col] - df["A_lag"]
    df["delta_B"] = df[factor_b_col] - df["B_lag"]

    # contribution of A changing (holding B at its previous value)
    df["contrib_A"] = df["delta_A"] * df["B_lag"]
    # contribution of B changing (holding A at its previous value)
    df["contrib_B"] = df["delta_B"] * df["A_lag"]
    # cross term (both changing together)
    df["contrib_cross"] = df["delta_A"] * df["delta_B"]

    return df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

factors_df["year_num"] = factors_df["year"].str.split("-").str[0].astype(int)
# apply the decomposition to your factors_df, for D3 and D5 separately
d3_decomp = decompose_product_change(factors_df, "D3_endow_factor", "D3_coef_factor")
d3_decomp = d3_decomp.rename(columns={
    "contrib_A": "D3_contrib_endow_change",
    "contrib_B": "D3_contrib_coef_change",
    "contrib_cross": "D3_contrib_cross"
})

d5_decomp = decompose_product_change(factors_df, "D5_endow_factor", "D5_coef_factor")
d5_decomp = d5_decomp.rename(columns={
    "contrib_A": "D5_contrib_endow_change",
    "contrib_B": "D5_contrib_coef_change",
    "contrib_cross": "D5_contrib_cross"
})

contrib_cols_d3 = ["D3_contrib_endow_change", "D3_contrib_coef_change", "D3_contrib_cross"]
contrib_cols_d5 = ["D5_contrib_endow_change", "D5_contrib_coef_change", "D5_contrib_cross"]
all_contrib_cols = contrib_cols_d3 + contrib_cols_d5

models_list = factors_df["model"].unique()
n_models = len(models_list)
jitter_span = 0.3
jitters = np.linspace(-jitter_span/2, jitter_span/2, n_models) if n_models > 1 else [0]

n_cols = 3
n_rows = -(-len(all_contrib_cols) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows), squeeze=False)
axes = axes.flatten()

decomp_lookup = {**{c: d3_decomp for c in contrib_cols_d3}, **{c: d5_decomp for c in contrib_cols_d5}}

for i, col in enumerate(all_contrib_cols):
    ax = axes[i]
    df_src = decomp_lookup[col]
    for model, jit in zip(models_list, jitters):
        sub = df_src[df_src["model"] == model].sort_values("year_num")
        ax.plot(sub["year_num"] + jit, sub[col], marker="o", label=model, alpha=0.85)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Year")
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title="Model", fontsize=8)

for j in range(len(all_contrib_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

factors_df["year_num"] = factors_df["year"].str.split("-").str[0].astype(int)
# apply the decomposition to your factors_df, for D3 and D5 separately
d3_decomp = decompose_product_change(factors_df, "D3_endow_factor", "D3_coef_factor")
d3_decomp = d3_decomp.rename(columns={
    "contrib_A": "D3_contrib_endow_change",
    "contrib_B": "D3_contrib_coef_change",
    "contrib_cross": "D3_contrib_cross"
})

d5_decomp = decompose_product_change(factors_df, "D5_endow_factor", "D5_coef_factor")
d5_decomp = d5_decomp.rename(columns={
    "contrib_A": "D5_contrib_endow_change",
    "contrib_B": "D5_contrib_coef_change",
    "contrib_cross": "D5_contrib_cross"
})

contrib_cols_d3 = ["D3_contrib_endow_change", "D3_contrib_coef_change", "D3_contrib_cross"]
contrib_cols_d5 = ["D5_contrib_endow_change", "D5_contrib_coef_change", "D5_contrib_cross"]
all_contrib_cols = contrib_cols_d3 + contrib_cols_d5

models_list = factors_df["model"].unique()
n_models = len(models_list)
jitter_span = 0.3
jitters = np.linspace(-jitter_span/2, jitter_span/2, n_models) if n_models > 1 else [0]

n_cols = 3
n_rows = -(-len(all_contrib_cols) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows), squeeze=False)
axes = axes.flatten()

decomp_lookup = {**{c: d3_decomp for c in contrib_cols_d3}, **{c: d5_decomp for c in contrib_cols_d5}}

for i, col in enumerate(all_contrib_cols):
    ax = axes[i]
    df_src = decomp_lookup[col]
    for model, jit in zip(models_list, jitters):
        sub = df_src[df_src["model"] == model].sort_values("year_num")
        ax.plot(sub["year_num"] + jit, sub[col], marker="o", label=model, alpha=0.85)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Year")
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title="Model", fontsize=8)

for j in range(len(all_contrib_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()